# 00_config


In [0]:
import os
import re
import uuid
import zipfile
import traceback
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
)

In [0]:
catalog = "iics"
schema  = "casemix"

control_table  = f"{catalog}.{schema}.zip_files_and_json_control_test"   # one row per source (NPHC and HSAR)
log_table      = f"{catalog}.{schema}.zip_files_execution_log_test"      # copy and unzip steps for both pipelines, plus the HSAR load step
load_log_table = f"{catalog}.{schema}.json_large_files_load_log"         # NPHC only: one row for every JSON file it loads

EXECUTED_BY = spark.sql("SELECT current_user()").first()[0]

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {control_table}")
# spark.sql(f"DROP TABLE IF EXISTS {log_table}")
# spark.sql(f"DROP TABLE IF EXISTS {load_log_table}")

### Zip files name extraction

In [0]:
# Zip (and JSON) names look like <name>_<start>_<MID>_<end>_<hhmmss>.zip, the same for NPHC and HSAR.
#   MID date -> the folder name under raw/ and processed/
#   END date -> used to check what is new (NPHC by month, HSAR by day)
# Example: case_data_001_20220605_20220620_20260101_235954.zip
#   gives mid_date = 20220620 and end_date = 20260101
FILE_NAME_PATTERN = re.compile(
    r"^(?P<table_name>.+?)_(?P<start_date>\d{8})_(?P<mid_date>\d{8})_(?P<end_date>\d{8})_\d{6}\.(zip|json)$",
    re.IGNORECASE,
)

### HSAR Settings

In [0]:
# JSON files inside the zips end with _<yyyymmdd>_<hhmmss>.json
# that last date becomes dt_data_file and drives the year / month partitions
HSAR_FILE_DATE_PATTERN = r"_(\d{8})_\d{6}\.[A-Za-z]+$"

HSAR_MAX_FILES_PER_TRIGGER = 500     # how many files Auto Loader reads per batch

### Control table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {control_table} (
    s_no                        INT NOT NULL,
    src_system                  STRING NOT NULL,   -- NPHC or HSAR: each pipeline reads only its own rows
    zip_source                  STRING NOT NULL,
    zip_source_path             STRING NOT NULL,   -- folder with the .zip files
    zip_tgt                     STRING NOT NULL,
    zip_tgt_path                STRING NOT NULL,   -- raw/        : copied zips, in <date>/ folders
    unzip_tgt_path              STRING NOT NULL,   -- processed/  : unzipped files, in <date>/<zip_name>/ folders
    file_type                   STRING NOT NULL,
    tgt_catalog                 STRING NOT NULL,
    tgt_schema                  STRING NOT NULL,
    tgt_table                   STRING NOT NULL,
    versioning_partition_filed  STRING NOT NULL,
    is_active                   INT NOT NULL
)
USING DELTA
""")

### Log tables
`log_table` keeps the copy and unzip records for both pipelines, plus one HSAR_LOAD record for each dated folder. `load_log_table` is for NPHC and has one row per JSON file, including the zip it came from.

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id",      StringType(),    True),
    StructField("stage",       StringType(),    True),   # COPY | UNZIP | HSAR_LOAD
    StructField("s_no",        IntegerType(),   True),
    StructField("source",      StringType(),    True),   # src_system
    StructField("file_name",   StringType(),    True),   # HSAR_LOAD: the dated folder
    StructField("source_path", StringType(),    True),
    StructField("target_path", StringType(),    True),   # HSAR_LOAD: the target table
    StructField("status",      StringType(),    True),   # SUCCESS | FAILED | SKIPPED
    StructField("message",     StringType(),    True),
    StructField("start_time",  TimestampType(), True),
    StructField("end_time",    TimestampType(), True),
])

LOAD_LOG_SCHEMA = StructType([
    StructField("run_id",           StringType(),    True),
    StructField("src",              StringType(),    True),
    StructField("src_path",         StringType(),    True),
    StructField("zip_file",         StringType(),    True),
    StructField("file_name",        StringType(),    True),
    StructField("file_path",        StringType(),    True),
    StructField("file_size_mb",     DoubleType(),    True),
    StructField("file_modified_ms", LongType(),      True),
    StructField("target_table",     StringType(),    True),
    StructField("status",           StringType(),    True),   # SUCCESS | FAILED
    StructField("error_type",       StringType(),    True),
    StructField("error_message",    StringType(),    True),
    StructField("started_at",       TimestampType(), True),
    StructField("ended_at",         TimestampType(), True),
    StructField("executed_by",      StringType(),    True),
])

def create_log_table(name, struct):
    cols = ",\n    ".join(f"{f.name} {f.dataType.simpleString().upper()}" for f in struct.fields)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} (\n    {cols}\n) USING DELTA")

create_log_table(log_table, LOG_SCHEMA)
create_log_table(load_log_table, LOAD_LOG_SCHEMA)

### Control rows

In [0]:
control_columns = """s_no, src_system, zip_source, zip_source_path, zip_tgt, zip_tgt_path, unzip_tgt_path,
                   file_type, tgt_catalog, tgt_schema, tgt_table, versioning_partition_filed, is_active"""

control_rows = [
    (1, "NPHC", "s3", "s3://cittabase-s3-test/Databricks-poc/zip_files/", "databricks_volume",
     "/Volumes/iics/casemix/history/zip_files/nphc/raw/",
     "/Volumes/iics/casemix/history/zip_files/nphc/processed/",
     "json", "iics", "casemix", "nphc_json_test", "file_name", 1),
    (2, "HSAR", "s3", "s3://cittabase-s3-test/Databricks-poc/hsar_zip_files/", "databricks_volume",
     "/Volumes/iics/casemix/history/zip_files/hsar/raw/",
     "/Volumes/iics/casemix/history/zip_files/hsar/processed/",
     "json", "iics", "casemix", "hsar_json_test", "n/a", 1),
]

for row in control_rows:
    already_there = spark.sql(f"SELECT 1 FROM {control_table} WHERE src_system = '{row[1]}'").count() > 0
    if not already_there:
        spark.sql(f"INSERT INTO {control_table} ({control_columns}) VALUES {row}")
        print(f"Added control row for {row[1]}")

In [0]:
%sql
SELECT * FROM iics.casemix.zip_files_and_json_control_test

In [0]:
%sql
-- DROP TABLE IF EXISTS iics.casemix.hsar_json_test

In [0]:
%sql
-- DROP TABLE IF EXISTS iics.casemix.nphc_json_test

In [0]:
%sql
select * from iics.casemix.json_large_files_load_log

In [0]:
%sql
select * from iics.casemix.zip_files_execution_log_test